In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import os
import glob
import re  # For removing parentheses


os.chdir('/Image/Directory/here/')
os.getcwd()
#Read collagen feature data
df_concat = pd.read_csv('collagen_global_properties_fractions.csv')
#Average data for each Leap ID
Leap_list=pd.unique(df_concat['Leap_ID'])
def patientavgs(patient_no,Patient_stats):
    #Mean_stats=Patient_stats.mean()
    Median_stats=Patient_stats.median()
    #SD_stats=Patient_stats.std()
    Patient_avgs=Median_stats#, Median_stats, SD_stats]
    return Patient_avgs
Patient_avgs_list=[]
Patient_variables=['Leap_ID']
k=0
for names in Leap_list:
    if names != 'LEAP084A':
        Patient_stats=df_concat[df_concat['Leap_ID']==names]
        Patient_avgs=[names]
        for i in range (0,9):
            Patient_avgs.append(patientavgs(names, Patient_stats[Patient_stats.columns[i]]))
            if k==0:
                Patient_variables.append(Patient_stats.columns[i])
        for j in range (10,21):   
            Patient_avgs.append(patientavgs(names, Patient_stats[Patient_stats.columns[j]]))
            if k==0:
                Patient_variables.append(Patient_stats.columns[j])
        k=k+1    
        Patient_avgs_list.append(Patient_avgs)
    
    
df_pdl=pd.DataFrame(Patient_avgs_list, columns=[Patient_variables])

In [ ]:
#Read patient metadata
df_metadata=pd.read_csv('CleanCohort_Metadata.csv')
df_metadata=pd.DataFrame(df_metadata)

In [ ]:
#concatenate collagen data and metadata
df_avgs_meta=pd.concat([df_pdl, df_metadata], axis=1)

In [ ]:
#Extract non-responder only data
df_avgs_Non_resp=df_avgs_meta[df_avgs_meta['Response']=='Non-Responder']


In [ ]:
#t-test
#Differences pre- vs post- treatment for all
from scipy import stats
a=df_avgs_meta[df_avgs_meta['Sample_Type_(pre/post treatment)']=='post']
b=df_avgs_meta[df_avgs_meta['Sample_Type_(pre/post treatment)']=='pre']
for i in range (1,21):
    a1=a[a.columns[i]]
    b1=b[b.columns[i]]
    t_test, pval=stats.ttest_ind(a1,b1,equal_var=False)
    print(a.columns[i],t_test,pval)

In [ ]:

# Filter the dataframe to only include 'pre' and 'post'
filtered_df = df_avgs_meta[df_avgs_meta['Sample_Type_(pre/post treatment)'].isin(['pre', 'post'])]
custom_palette = {'pre':'#c9a300', 'post': 'purple'}
for i in range(1, 21):
    plt.figure(figsize=(8, 6))  # Create a new figure for each plot

    y_col = df_avgs_meta.columns[i]
    clean_label = ' '.join(y_col) # Remove parentheses from label
    # Swarmplot
    sns.swarmplot(data=filtered_df,
                  x='Sample_Type_(pre/post treatment)',
                  y=y_col,
                  hue='Sample_Type_(pre/post treatment)',
                  palette=custom_palette
                  #dodge=True)
                  )
    # Boxplot overlaid
    sns.boxplot(data=filtered_df,
                x='Sample_Type_(pre/post treatment)',
                y=y_col,
                showcaps=False,
                boxprops={'facecolor': 'None'},
                showfliers=False,
                palette=custom_palette)
    plt.ylabel(clean_label)
    plt.tight_layout()
    #plt.show()

In [ ]:
#Differences pre - vs post-treatment Non-Responders only
a=df_avgs_Non_resp[df_avgs_Non_resp['Sample_Type_(pre/post treatment)']=='post']
b=df_avgs_Non_resp[df_avgs_Non_resp['Sample_Type_(pre/post treatment)']=='pre']
for i in range (1,21):
    a1=a[a.columns[i]]
    b1=b[b.columns[i]]
    t_test, pval=stats.ttest_ind(a1,b1,equal_var=False)
    print(a.columns[i],t_test,pval)

In [ ]:
#Example of plot for pre-vs-post-treatment Non-responders: curvature
for i in range (12,13):
    sns.set(font_scale=1.5)
    custom_palette = {'pre':'#c9a300', 'post': 'purple'}
    # plot swarmplot
    ax = sns.swarmplot(df_avgs_Non_resp,x ='Sample_Type_(pre/post treatment)', 
                       y = df_avgs_Non_resp.columns[i],
                       hue = 'Sample_Type_(pre/post treatment)',
                       palette=custom_palette,
                       size=10)
    
    ax.set_facecolor('w')
    #plot boxplot
    sns.boxplot(x='Sample_Type_(pre/post treatment)', y=df_avgs_Non_resp.columns[i], data=df_avgs_Non_resp, 
                 showcaps=False,boxprops={'facecolor':'None'},
                 palette=custom_palette,
                 showfliers=False, ax=ax)
    plt.xlabel('pre- / post- treatment')
    plt.ylabel('Fibre curvature (degrees)')
    plt.ylim(40,80)
    x1, x2 = 0,1  
    y, h, col = 75, 1, 'k'

    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
    plt.text((x1+x2)*.5, y+h, "*", ha='center', va='bottom', color=col, fontsize=25)
    for spine in ax.spines.values():
        spine.set_visible(True)   # Ensure spines are visible
        spine.set_linewidth(1.5)  # Optional: make the border thicker
        spine.set_color('black')  # Optional: set border color
    plt.tight_layout()
    plt.savefig('NS_Pre_vs_post_treatment_curvature.png')

In [ ]:
#Example of plot for pre-vs-post-treatment Non-responders: anisotropy
for i in range (15,16):
    sns.set(font_scale=1.5)
    custom_palette={'pre':'#c9a300', 'post': 'purple'}
    # plot swarmplot
    ax = sns.swarmplot(df_avgs_Non_resp,x ='Sample_Type_(pre/post treatment)', 
                       y = df_avgs_Non_resp.columns[i],
                       hue = 'Sample_Type_(pre/post treatment)',
                       palette=custom_palette,
                       size=10) 
    ax.set_facecolor('w')
    # plot boxplot
    sns.boxplot(x='Sample_Type_(pre/post treatment)', y=df_avgs_Non_resp.columns[i], data=df_avgs_Non_resp, 
                 showcaps=False,boxprops={'facecolor':'None'},
                 palette=custom_palette,
                 showfliers=False, ax=ax)
    plt.xlabel('pre- / post- treatment')
    plt.ylabel('Anisotropy')
    plt.ylim(0,0.8)
    x1, x2 = 0,1  
    y, h, col = 0.7, 0.01, 'k'

    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
    plt.text((x1+x2)*.5, y+h, "**", ha='center', va='bottom', color=col, fontsize=25)
    for spine in ax.spines.values():
        spine.set_visible(True)   # Ensure spines are visible
        spine.set_linewidth(1.5)  # Optional: make the border thicker
        spine.set_color('black')  # Optional: set border color
    plt.tight_layout()
    plt.savefig('NS_Pre_vs_post_treatment_anisotropy.png')

In [ ]:
#Split data into pre- and post-treatment
df_avgs_post=df_avgs_meta[df_avgs_meta['Sample_Type_(pre/post treatment)']=='post']
df_avgs_pre=df_avgs_meta[df_avgs_meta['Sample_Type_(pre/post treatment)']=='pre']


In [ ]:
#Non responders only: differences between treatment groups
df_avgs_NonResp_post=df_avgs_post[df_avgs_post['Response']=='Non-Responder']

a=df_avgs_NonResp_post[df_avgs_NonResp_post['NACT_Treatment _Group']=='EC-T']
b=df_avgs_NonResp_post[df_avgs_NonResp_post['NACT_Treatment _Group']=='EC-T-carbo']

for i in range (1,21):
    a1=a[a.columns[i]]
    b1=b[b.columns[i]]
    t_test, pval=stats.ttest_ind(a1,b1,equal_var=False)
    print(a.columns[i],t_test,pval)

In [ ]:
#Example of plot for supplemental figure S6-2, comparing treatment groups (Carboplatin + ECT vs ECT alone)
#for non-responders only.
for i in range (3,4):
    ax = sns.swarmplot(df_avgs_NonResp_post,x ='NACT_Treatment _Group', y = df_avgs_NonResp_post.columns[i],hue = 'NACT_Treatment _Group')#,common_norm=False)
    # plot swarmplot

    sns.boxplot(x='NACT_Treatment _Group', y=df_avgs_NonResp_post.columns[i], data=df_avgs_NonResp_post, 
                 showcaps=False,boxprops={'facecolor':'None'},
                 showfliers=False, ax=ax)
    plt.ylabel('Perimeter (μm)')
    #plt.figure()
    #plt.ylim(0,0.5)
    x1, x2 = 0,1  
    y, h, col = 0.45, 0.01, 'k'

    #plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
    #plt.text((x1+x2)*.5, y+h, "*", ha='center', va='bottom', color=col, fontsize=25)
    plt.tight_layout()
    plt.savefig('Non_respPostTreatment_treatment_group_Perimeter.png')

In [ ]:
#Pre-treatment only. Extreme responders vs non-extreme
a=df_avgs_pre[df_avgs_pre['Extreme_NR']==False]
b=df_avgs_pre[df_avgs_pre['Extreme_NR']==True]

for i in range (1,21):
    a1=a[a.columns[i]]
    b1=b[b.columns[i]]
    t_test, pval=stats.ttest_ind(a1,b1,equal_var=False)
    print(a.columns[i],t_test,pval)

In [ ]:
#Example of plot for Extreme responders vs non-extreme in pre-treament only
for i in range (1,2):
    sns.set(font_scale=1.5)
    custom_palette={'#008080','#b7410e' }
    # plot swarmplot
    ax = sns.swarmplot(df_avgs_pre,x ='Extreme_NR', 
                       y = df_avgs_pre.columns[i],
                       hue = 'Extreme_NR',
                       palette=custom_palette,
                       legend=False,
                       size=10) 
    ax.set_facecolor('w')
    # plot boxplot
    sns.boxplot(x='Extreme_NR', y=df_avgs_pre.columns[i], data=df_avgs_pre, 
                 showcaps=False,boxprops={'facecolor':'None'},
                 palette=custom_palette,
                 showfliers=False, ax=ax)
    plt.legend([],[], frameon=False)
    plt.xlabel('Extreme non-responder')
    plt.ylabel('Area')
    plt.ylim(0,0.6)
    x1, x2 = 0,1  
    y, h, col = 0.7, 0.01, 'k'

   # plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
    #plt.text((x1+x2)*.5, y+h, "**", ha='center', va='bottom', color=col, fontsize=25)
    for spine in ax.spines.values():
        spine.set_visible(True)   # Ensure spines are visible
        spine.set_linewidth(1.5)  # Optional: make the border thicker
        spine.set_color('black')  # Optional: set border color
    plt.tight_layout()
    plt.savefig('Pre_treat_ENS_vs_notENS_Area_python.png')

In [ ]:
for i in range (2,3):
    sns.set(font_scale=1.5)
    custom_palette={'#008080','#b7410e' }
    # plot swarmplot
    ax = sns.swarmplot(df_avgs_pre,x ='Extreme_NR', 
                       y = df_avgs_pre.columns[i],
                       palette=custom_palette,
                       hue = 'Extreme_NR',
                       size=10) 
    ax.set_facecolor('w')
    # plot boxplot
    sns.boxplot(x='Extreme_NR', y=df_avgs_pre.columns[i], data=df_avgs_pre, 
                 showcaps=False,boxprops={'facecolor':'None'},
                 palette=custom_palette,
                 showfliers=False, ax=ax)
    plt.legend([],[], frameon=False)
    plt.xlabel('Extreme non-responder')
    plt.ylabel('Compactness')
    plt.ylim(0,1.5)
    x1, x2 = 0,1  
    y, h, col = 0.7, 0.01, 'k'

   # plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
    #plt.text((x1+x2)*.5, y+h, "**", ha='center', va='bottom', color=col, fontsize=25)
    for spine in ax.spines.values():
        spine.set_visible(True)   # Ensure spines are visible
        spine.set_linewidth(1.5)  # Optional: make the border thicker
        spine.set_color('black')  # Optional: set border color
    plt.tight_layout()
    plt.savefig('Pre_treat_ENS_vs_notENS_compactness.png')

In [ ]:
for i in range (2,3):
    sns.set(font_scale=1.5)
    custom_palette={'#008080','#b7410e' }
    # plot swarmplot
    ax = sns.swarmplot(df_avgs_pre,x ='Extreme_NR', 
                       y = df_avgs_pre.columns[i],
                       palette=custom_palette,
                       hue = 'Extreme_NR',
                       size=10) 
    ax.set_facecolor('w')
    # plot boxplot
    sns.boxplot(x='Extreme_NR', y=df_avgs_pre.columns[i], data=df_avgs_pre, 
                 showcaps=False,boxprops={'facecolor':'None'},
                 palette=custom_palette,
                 showfliers=False, ax=ax)
    plt.legend([],[], frameon=False)
    plt.xlabel('Extreme non-responder')
    plt.ylabel('Compactness')
    plt.ylim(0,1.5)
    x1, x2 = 0,1  
    y, h, col = 0.7, 0.01, 'k'

   # plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
    #plt.text((x1+x2)*.5, y+h, "**", ha='center', va='bottom', color=col, fontsize=25)
    for spine in ax.spines.values():
        spine.set_visible(True)   # Ensure spines are visible
        spine.set_linewidth(1.5)  # Optional: make the border thicker
        spine.set_color('black')  # Optional: set border color
    plt.tight_layout()
    plt.savefig('Pre_treat_ENS_vs_notENS_compactness.png')

In [ ]:
for i in range (12,13):
    sns.set(font_scale=1.5)
    custom_palette={'#008080','#b7410e' }
    # plot swarmplot
    ax = sns.swarmplot(df_avgs_pre,x ='Extreme_NR', 
                       y = df_avgs_pre.columns[i],
                       hue = 'Extreme_NR',
                       palette=custom_palette,
                       size=10) 
    ax.set_facecolor('w')
    # plot boxplot
    sns.boxplot(x='Extreme_NR', y=df_avgs_pre.columns[i], data=df_avgs_pre, 
                 showcaps=False,boxprops={'facecolor':'None'},
                 palette=custom_palette,
                 showfliers=False, ax=ax)
    plt.legend([],[], frameon=False)
    plt.xlabel('Extreme non-responder')
    plt.ylabel('Fibre curvature (degrees)')
    plt.ylim(30,85)
    x1, x2 = 0,1  
    y, h, col = 75, 1, 'k'

    plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
    plt.text((x1+x2)*.5, y+h, "*", ha='center', va='bottom', color=col, fontsize=25)
    for spine in ax.spines.values():
        spine.set_visible(True)   # Ensure spines are visible
        spine.set_linewidth(1.5)  # Optional: make the border thicker
        spine.set_color('black')  # Optional: set border color
    plt.tight_layout()
    plt.savefig('Pre_treat_ENS_vs_notENS_curvature.png')

In [ ]:
#Example of plot for responders vs non-responders in pre-treament only
for i in range (7,8):
    sns.set(font_scale=1.5)
    custom_palette={'Non-Responder': 'maroon', 'Responder': 'darkgreen'}
    # plot swarmplot
    ax = sns.swarmplot(df_avgs_pre,x ='Response', 
                       y = df_avgs_pre.columns[i],
                       palette=custom_palette,
                       hue = 'Response',
                       size=10) 
    ax.set_facecolor('w')
    # plot boxplot
    sns.boxplot(x='Response', y=df_avgs_pre.columns[i], data=df_avgs_pre, 
                 showcaps=False,boxprops={'facecolor':'None'},
                 palette=custom_palette,
                 showfliers=False, ax=ax)
    plt.legend([],[], frameon=False)
    plt.xlabel('Response')
    plt.ylabel('Circular variance')
    #plt.ylim(0,0.6)
    #x1, x2 = 0,1  
    #y, h, col = 0.7, 0.01, 'k'

   # plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
    #plt.text((x1+x2)*.5, y+h, "**", ha='center', va='bottom', color=col, fontsize=25)
    for spine in ax.spines.values():
        spine.set_visible(True)   # Ensure spines are visible
        spine.set_linewidth(1.5)  # Optional: make the border thicker
        spine.set_color('black')  # Optional: set border color
    plt.tight_layout()
    plt.savefig('Pre_treat_responder_vs_noresponder_circular_var.png')

In [ ]:
for i in range (12,13):
    sns.set(font_scale=1.5)
    custom_palette={'Non-Responder': 'maroon', 'Responder': 'darkgreen'}
    # plot swarmplot
    ax = sns.swarmplot(df_avgs_pre,x ='Response', 
                       y = df_avgs_pre.columns[i],
                       hue = 'Response',
                       palette=custom_palette,
                       size=10) 
    ax.set_facecolor('w')
    # plot boxplot
    sns.boxplot(x='Response', y=df_avgs_pre.columns[i], data=df_avgs_pre, 
                 showcaps=False,boxprops={'facecolor':'None'},
                 palette=custom_palette,
                 showfliers=False, ax=ax)
    plt.legend([],[], frameon=False)
    plt.xlabel('Response')
    plt.ylabel('Fibre curvature (degrees)')
    #plt.ylim(0,0.6)
    #x1, x2 = 0,1  
    #y, h, col = 0.7, 0.01, 'k'

   # plt.plot([x1, x1, x2, x2], [y, y+h, y+h, y], lw=1.5, c=col)
    #plt.text((x1+x2)*.5, y+h, "**", ha='center', va='bottom', color=col, fontsize=25)
    for spine in ax.spines.values():
        spine.set_visible(True)   # Ensure spines are visible
        spine.set_linewidth(1.5)  # Optional: make the border thicker
        spine.set_color('black')  # Optional: set border color
    plt.tight_layout()
    plt.savefig('Pre_treat_responder_vs_noresponder_curvature.png')

In [ ]:
a=df_avgs_pre[df_avgs_pre['Response']=='Responder']
b=df_avgs_pre[df_avgs_pre['Response']=='Non-Responder']
for i in range (1,21):
    a1=a[a.columns[i]]
    b1=b[b.columns[i]]
    t_test, pval=stats.ttest_ind(a1,b1,equal_var=False)
    print(a.columns[i],t_test,pval)

In [ ]:
#Calculate standard deviation of LEAP ID samples
df_concat = pd.read_csv('collagen_global_properties_fractions.csv')
Leap_list=pd.unique(df_concat['Leap_ID'])
def patientavgs(patient_no,Patient_stats):
    #Mean_stats=Patient_stats.mean()
    #Median_stats=Patient_stats.median()
    SD_stats=Patient_stats.std()
    Patient_sd=SD_stats
    return Patient_sd
Patient_sd_list=[]
Patient_variables=['Leap_ID']
k=0
for names in Leap_list:
    if names != 'LEAP084A':
        Patient_stats=df_concat[df_concat['Leap_ID']==names]
        Patient_sd=[names]
        for i in range (0,9):
            Patient_sd.append(patientavgs(names, Patient_stats[Patient_stats.columns[i]]))
            if k==0:
                Patient_variables.append(Patient_stats.columns[i])
        for j in range (10,21):   
            Patient_sd.append(patientavgs(names, Patient_stats[Patient_stats.columns[j]]))
            if k==0:
                Patient_variables.append(Patient_stats.columns[j])
        k=k+1    
        Patient_sd_list.append(Patient_sd)
    
    
df_pdl_sd=pd.DataFrame(Patient_sd_list, columns=[Patient_variables])


In [ ]:
df_pdl_sd.to_csv('collagen_feature_sd.csv')

In [ ]:
#Plots of collagen features over cycles for patients with multiple sets of FORCE cycle data. 
df = pd.read_csv('Collagen_feature_averages_frozen_samples_only.csv')

patient_ID_vals = [49, 50, 53, 57]
custom_order = ['PreNACT', 'Cycle1', 'Cycle2']
x_map = {cycle: idx for idx, cycle in enumerate(custom_order)}

colors = ['#1b9e77', '#d95f02', '#7570b3', '#e7298a']  # professional palette
markers = ['o', 's', 'D', '^']
linestyles = ['-', '--', '-.', ':']

for fig_idx, i in enumerate(range(15, 32)):
    plt.figure(figsize=(6, 4))
    y_col = df.columns[i]
    err_col = df.columns[i + 17]

    for idx, (pid, color, marker, base_linestyle) in enumerate(zip(patient_ID_vals, colors, markers, linestyles)):
        df_patient = df[df['Patient_ID'] == pid]

        grouped = df_patient.groupby('FORCE_cycle').agg({
            y_col: 'mean',
            err_col: 'mean'
        })

        # Drop missing data and sort by correct x-axis order
        grouped = grouped.loc[grouped.index.intersection(custom_order)]
        grouped = grouped.sort_index(key=lambda x: [x_map[val] for val in x])

        if grouped.empty:
            continue

        # Map to numeric x-axis values
        x_vals = [x_map[fc] for fc in grouped.index]
        y_vals = grouped[y_col].values
        y_errs = grouped[err_col].values

        # Determine if timepoints are consecutive
        is_consecutive = all((j - i == 1) for i, j in zip(x_vals, x_vals[1:]))

        plt.errorbar(
            x=x_vals, y=y_vals, yerr=y_errs,
            label=f'Patient {pid}' if fig_idx == 0 else None,
            color=color, marker=marker,
            linestyle=base_linestyle if is_consecutive else 'dashed',
            linewidth=1.5, markersize=6, capsize=4, alpha=0.9
        )

    plt.xticks(ticks=range(len(custom_order)), labels=custom_order, fontsize=10)
    plt.ylabel(y_col, fontsize=11)
    plt.title(f'Feature: {y_col}', fontsize=12)
    if fig_idx == 0:
        plt.legend(title='Patient ID', fontsize=9, title_fontsize=10, frameon=False)
    plt.grid(alpha=0.2)
    plt.tight_layout()
    ftype='.png'
    filename=(y_col + ftype)
    print(filename)
    plt.savefig(filename)